# Level 2 · Continual learning

Level 1 produced one number per benchmark: how your method does under the **final** harness.
This notebook is about the question that number cannot answer — *what happened on the way there*.

The benchmark's premise is that a harness **grows**. Each domain arrives in stages, and every
stage adds tools, skills or specialists to the catalog and brings its own cohort of tasks. So
there are two populations moving at once: the capabilities available, and the tasks being asked.
Crossing them gives the object everything here is built on.

### The performance matrix

\[ P_{t,\tau} = \text{accuracy on cohort } \tau \text{ evaluated under harness stage } t \]

It is **lower-triangular**, because cohort \(\tau\) does not exist until stage \(\tau\) is reached.
Three different questions read three different parts of it:

|   | where | question |
|---|---|---|
| **Adaptation** | the diagonal \(P_{t,t}\) | can the method use capabilities the moment they appear? |
| **Retention** | below the diagonal \(P_{t,\tau},\ \tau<t\) | does it still solve old tasks once the catalog is bigger? |
| **ACC** | the last row, pooled | the headline Level 1 reported |

```
                 cohort
                   v1      v2      v3
  under H_v1     0.31       ·       ·      <- adaptation: v1 tasks, v1 tools
  under H_v2     0.28    0.40       ·      <- retention:  v1 tasks, but v2 tools now exist
  under H_v3     0.25    0.36    0.22      <- the last row is what ACC pools
```

Reading down a column is the story of one cohort as the world grows around it. `0.31 → 0.28 →
0.25` on cohort v1 is **forgetting** — nothing about those tasks changed, only the harness did.
That column delta is what **BWT** summarizes. **FWT** is the other delta: on the diagonal,
comparing a cohort under its own harness *before* and *after* that stage's adaptation.

### Two ways a method meets a growing harness

| `mode=` | what it means | needs |
|---|---|---|
| `"deployment_eval"` | the method is **fixed**; only the harness grows around it | nothing — any `agent(task)` |
| `"self_evolving_adapt_eval"` | the method also **learns** at each stage, from that stage's training split | an `adapt(stage, tasks)` step |

Deployment isolates the harness's effect. Self-evolving asks whether learning keeps up with it.
BWT is meaningful in both; **FWT only exists in self-evolving**, because it measures adaptation.

### The three tutorials

| | | |
|---|---|---|
| **1 · Leaderboard your agent** | your method → one comparable row | [Colab](https://colab.research.google.com/drive/1xJEpRf_s0zG-M9QynS3MBk7Nkr-xB11r) |
| **2 · Continual learning** | the matrix, retention, BWT and FWT | [Colab](https://colab.research.google.com/drive/1vrcGelN9GmwiCK25c6qZNG3Z0sHWxE5o) ← **you are here** |
| **3 · Harnesses & modes** | the provided agents, every mode, standard non-evolving eval, full reference | [Colab](https://colab.research.google.com/drive/1mYQEDCVFStXMRWYI2hpEGBXwyRx1NSFj) |

Setup and the health check are identical to Level 1 — run them, then start at §1.

In [ ]:
# Setup, in one cell. The client SDK is served BY the service (GET /sdk), so there is no
# PyPI account, no repo checkout, and no service URL anywhere in your code afterwards.
import getpass, importlib, json, os, re, subprocess, sys, tempfile, urllib.error, urllib.request
from functools import partial

SERVICE_URL = os.environ.get("EVAL_SERVICE_URL",
                             "https://educator-marrow-cultural.ngrok-free.dev")

if not os.environ.get("EVAL_SERVICE_API_KEY"):
    os.environ["EVAL_SERVICE_API_KEY"] = getpass.getpass("Eval service key (MyAuthtoken): ")
SDK_HEADERS = {"Authorization": f"Bearer {os.environ['EVAL_SERVICE_API_KEY']}",
               "ngrok-skip-browser-warning": "true"}
def sdk_read(request):
    try:
        with urllib.request.urlopen(request) as response:
            return response.read()
    except urllib.error.HTTPError as exc:
        try: detail = json.loads(exc.read()).get("detail", str(exc))
        except Exception: detail = str(exc)
        raise SystemExit(f"SDK download denied: {detail}") from None


def _sdk_current(minimum=(0, 14, 0)) -> bool:
    # A stale pre-installed copy would shadow the service's wheel, so check the API and
    # version rather than mere importability.
    try:
        import simple_agentic_evals as m
        v = tuple(int(x) for x in str(getattr(m, "__version__", "0")).split(".")[:3])
        return hasattr(m, "run_benchmark") and v >= minimum
    except Exception:
        return False


subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openai"], check=True)
if not _sdk_current():
    req = urllib.request.Request(f"{SERVICE_URL}/sdk", headers=SDK_HEADERS)
    manifest = json.loads(sdk_read(req))
    wheel = os.path.join(tempfile.mkdtemp(prefix="eval_sdk_"), manifest["filename"])
    wheel_req = urllib.request.Request(f"{SERVICE_URL}{manifest['path']}", headers=SDK_HEADERS)
    with open(wheel, "wb") as output:
        output.write(sdk_read(wheel_req))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade",
                    "--force-reinstall", "--no-deps", wheel], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "httpx"], check=True)
    for k in [k for k in list(sys.modules) if k.startswith("simple_agentic_evals")]:
        del sys.modules[k]
    importlib.invalidate_caches()

if not os.environ.get("OPENAI_API_KEY"):        # the key YOUR agent thinks with
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")

from openai import OpenAI
from simple_agentic_evals import (EvalClient, ServiceError, run_benchmark,
                                  react_agent, acp_codex_agent, to_openai_tools)

client    = EvalClient()     # the endpoint is baked into the wheel -- no URL to paste
oai       = OpenAI()         # only for the bring-your-own-agent examples
LLM_MODEL = os.environ.get("EVAL_LLM_MODEL", "gpt-4o-mini")
# Every /v1/* route needs the key, health included -- a rejected key means you
# cannot use this service, so it should not report itself healthy. So this one
# call both connects and proves the key, and a bad key fails HERE with the
# message that says where to get a new one.
try:
    _h = client.health()
except ServiceError as e:
    raise SystemExit(f"cannot use the eval service: {e.detail}") from None
print("connected ->", _h)


def harness(fn, *args, **kwargs):
    """Call a provided harness, or say why this deployment can't and return None.

    react_agent / acp_codex_agent execute ON THE SERVICE, so they depend on what that host
    has installed. Bringing your own agent needs none of it.
    """
    try:
        return fn(*args, **kwargs)
    except ServiceError as e:
        # The service prefixes a banner and appends the subprocess's stderr tail, so the
        # LAST line is the actual cause -- the first is just "...stderr tail:".
        lines = [ln for ln in (e.detail or "").strip().splitlines() if ln.strip()]
        print(f"  [harness unavailable] {e.status_code}: {(lines[-1] if lines else e)!s:.170}")
        if len(lines) > 1:
            print(f"     ({len(lines)} more lines of server traceback in e.detail)")
        return None

## 0. Health check — run this first

The service is a **deployment**, not a library. The environment, the graders and the two
provided harnesses all execute on *that host*, so what actually works depends on what it has
installed. The failure mode is quiet: an unavailable harness returns **without acting**, and
its run then scores the do-nothing baseline — which reads as a real (bad) result rather than
an error. Verify the deployment before you trust a number from it.

Each row is one capability, and a red row tells you what you lose, not what you did wrong:

| row | what it proves | if it's red |
|---|---|---|
| `reachable`, `sdk` | service is up; your client matches its served wheel | nothing works / re-run Setup |
| `catalog`, `eog tasks` | both benchmarks advertised, the slice is populated *and its count matches the catalog* | that data isn't deployed |
| `ale tasks` | the ALE **denominator**: exactly the tasks a published number counts | you can't compare an ALE score to anything |
| `eog env+grade` | session provisioning, the gym MCP proxy, SQL verifiers | all of EOG (§2) |
| `resources` | the evolving axis — `oracle` ⊆ `accumulative` | the evolving modes (§6) |
| `react`, `codex` | the **server-side** harnesses, and the host packages they need | those harnesses only — your own loop (§2) needs neither |
| `ale` | input staging, artifact submit, the task's own `evaluate()` | all of ALE (§5) |

The harness rows are the only way to spot a server missing `langchain` or the Codex CLI. A
harness that *runs* but stops with `error` is counted **red** — that's the silent failure this
section exists to catch. Set `DEEP = False` to skip the three rows that cost LLM calls or boot
a sandbox.

The task-count rows are about a subtler kind of wrong. ALE ships 152 tasks, but 47 of them need
a non-Linux VM and a few more die in their own loader on any given host, so only a subset can
produce a number at all — and a task that provisions and returns `0.0` for an environmental
reason is indistinguishable from an agent that tried and failed. The service therefore serves
only the tasks its exclusion manifest says are measurable, and `ale tasks` re-derives that
count from the catalog and fails if the two disagree. A green row means an ALE score you can
put next to someone else's.

In [ ]:
import urllib.error
DEEP = True    # False -> skip the rows that cost LLM calls or boot an ALE sandbox

# Fixed reference slices, so this stays a self-test of the DEPLOYMENT whatever you set in 1.
EOG_PROBE = ("evovling_tools", "eog", 1, "test", "hr")   # dataset, benchmark, version, split, domain
ALE_PROBE = ("evovling_tools", "ale", 1, "legal/agora_governance_classify_instance_1")
_rows = []


def _check(name, fn, lose="", deep=False):
    if deep and not DEEP:
        _rows.append((name, "SKIP"))
        print(f"  [SKIP] {name:14} DEEP=False")
        return
    try:
        detail, status = fn(), "OK"
    except ServiceError as e:                    # the service said no -- report why
        lines = [ln for ln in (e.detail or "").strip().splitlines() if ln.strip()]
        detail, status = f"{e.status_code}: {(lines[-1] if lines else e)!s:.100}", "FAIL"
    except Exception as e:
        detail, status = f"{type(e).__name__}: {e!s:.100}", "FAIL"
    _rows.append((name, status))
    print(f"  [{status:4}] {name:14} {detail}")
    if status == "FAIL" and lose:
        print(f"         lose: {lose}")


def _one_eog():
    return next(client.tasks(*EOG_PROBE, limit=1))


def _auth():
    """Prove the key works before anything expensive depends on it.

    A rejected key fails every row below this one, all with errors that look like
    the deployment is broken rather than like a credential problem -- so name it here.
    """
    req = urllib.request.Request(f"{SERVICE_URL}/v1/health",
                                 headers={"ngrok-skip-browser-warning": "true"})
    try:                                          # no key at all: is the gate even on?
        urllib.request.urlopen(req, timeout=30)
        gated = False
    except urllib.error.HTTPError as e:
        if e.code not in (401, 403, 503):
            raise
        gated = True
    client.benchmarks()                           # now WITH the key -- raises if rejected
    return ("key accepted" if gated else
            "key accepted (deployment is currently open -- no key required)")


def _service():
    h = client.health()
    if not h.get("ok"):
        raise RuntimeError(f"health.ok={h.get('ok')!r}")
    return f"ok, {h.get('active_sessions')} live session(s), ttl {h.get('ttl_sec')}s"


def _sdk():
    import simple_agentic_evals as m
    req = urllib.request.Request(f"{SERVICE_URL}/sdk",
                                 headers={"Authorization": f"Bearer {os.environ['EVAL_SERVICE_API_KEY']}",
                                          "ngrok-skip-browser-warning": "true"})
    served = re.search(r"-(\d+\.\d+\.\d+)-",
                       json.loads(sdk_read(req))["filename"]).group(1)
    mine = str(getattr(m, "__version__", "?"))
    if mine != served:                           # a stale client drifts from the API silently
        raise RuntimeError(f"client {mine} != served {served} -- re-run Setup")
    return f"{mine}, matches the served wheel"


def _catalog():
    kinds = {b["kind"] for ds in client.benchmarks()["datasets"]
             for b in ds["benchmarks"] if b.get("kind") in ("eog", "ale")}
    if {"eog", "ale"} - kinds:
        raise RuntimeError(f"catalog is missing {sorted({'eog', 'ale'} - kinds)}")
    return "eog + ale both advertised"


def _advertised(dataset, benchmark, version, split, domain):
    """What the catalog claims a stage holds, for cross-checking against task_ids."""
    for ds in client.benchmarks()["datasets"]:
        if ds["dataset"] != dataset:
            continue
        for b in ds["benchmarks"]:
            if b["benchmark"] != benchmark:
                continue
            for d in b["domains"]:
                if d["domain"] != domain:
                    continue
                for v in d["versions"]:
                    if v["version"] == version:
                        return v[f"n_{split}"]
    return None


def _eog_tasks():
    ids = client.task_ids(*EOG_PROBE)
    if not ids:
        raise RuntimeError("reference slice is empty")
    n = _advertised(*EOG_PROBE)                  # catalog and listing must agree
    if n != len(ids):
        raise RuntimeError(f"catalog advertises {n} tasks, task_ids returns {len(ids)}")
    return f"{len(ids)} tasks in {'/'.join(map(str, EOG_PROBE[1:]))}, matches the catalog"


def _ale_tasks():
    """ALE serves only what a published number can count -- verify the arithmetic."""
    m = client.health().get("ale_tasks") or {}
    if not m.get("ok"):
        raise RuntimeError(f"ALE task set unavailable: {m.get('reason')}")
    if not m.get("manifest_agrees"):
        raise RuntimeError(f"manifest drifted from ALE's task lists: {m.get('disagreements')}")
    if not m.get("filtered"):
        raise RuntimeError("this deployment sets EVAL_SERVICE_ALE_SERVE_ALL, so the catalog "
                           "includes tasks no published ALE number counts")
    served = set()
    for ds in client.benchmarks()["datasets"]:
        for b in ds["benchmarks"]:
            if b.get("kind") != "ale":
                continue
            for d in b["domains"]:
                for v in [x["version"] for x in d["versions"]]:
                    for sp in ("train", "test"):
                        served |= set(client.task_ids(ds["dataset"], "ale", v, sp,
                                                      d["domain"] or None))
    if len(served) != m["n_runnable"]:
        raise RuntimeError(f"catalog lists {len(served)} ale tasks, "
                           f"the manifest says {m['n_runnable']} are runnable")
    if m.get("example_filtered") in served:      # prove the filter is actually live
        raise RuntimeError(f"withheld task {m['example_filtered']} is still listed")
    return (f"{len(served)}/{m['n_suite']} runnable = {m['n_docker_support']} docker + "
            f"{m['n_privileged']} privileged; {m['n_excluded']} excluded + "
            f"{m['n_not_linux']} non-Linux withheld")


def _eog_env():
    task = _one_eog()
    with task:                                   # fresh DB, freed on exit
        mcp = task.mcp_session(task.mcp_servers[0])
        try:
            tools = mcp.list_tools()
        finally:
            mcp.close()
        if not tools:
            raise RuntimeError("gym returned 0 MCP tools")
        g = task.grade(keep_alive=True)          # no agent acted -> this is the baseline
    if not g.n_total:
        raise RuntimeError("grader ran 0 verifiers")
    return f"{len(tools)} tools, {g.n_total} verifiers, 1-task baseline {g.pass_rate:.2f}"


def _resources():
    tid = client.task_ids(*EOG_PROBE)[0]
    c = {m: client.resources(*EOG_PROBE[:3], task_id=tid, split=EOG_PROBE[3],
                             domain=EOG_PROBE[4], mode=m)["count"]
         for m in ("none", "oracle", "accumulative")}
    if c["accumulative"] < c["oracle"]:
        raise RuntimeError(f"accumulative {c['accumulative']} < oracle {c['oracle']}")
    return "  ".join(f"{k}={v}" for k, v in c.items())


def _harness_row(fn, **kw):
    task = _one_eog()
    with task:
        run = fn(task, api_key=os.environ["OPENAI_API_KEY"], **kw)
    stopped = getattr(run, "stopped", "?")
    if stopped == "error":                       # ran, but the turn failed on the server
        raise RuntimeError("harness returned stopped='error' -- see the service log")
    return f"ran on the service (stopped={stopped})"


def _ale():
    ad = client.health().get("ale_docker") or {}
    task = client.task(*ALE_PROBE, domain=None)  # ALE is flat -> domain=None
    with task:
        files = task.inputs()
        if not files:
            raise RuntimeError("no input files staged")
        task.fetch_input(files[0]["path"])
        # A stub artifact is enough: we're proving evaluate() executes, not scoring well.
        task.submit_text(task.output_path or "output/agent_output.json", "{}")
        g = task.grade(keep_alive=True)
    return (f"{len(files)} inputs, evaluate() ran (stub -> {g.pass_rate}), "
            f"sandbox={ad.get('enabled')} dind={ad.get('dind_available')}")


print("deployment:", SERVICE_URL)
_check("auth",          _auth,      "everything -- get a key from MyAuthtoken")
_check("reachable",     _service,   "everything")
_check("sdk",           _sdk,       "silent API drift -- re-run Setup")
_check("catalog",       _catalog,   "a benchmark isn't deployed here")
_check("eog tasks",     _eog_tasks, "this EOG slice")
_check("ale tasks",     _ale_tasks, "a trustworthy ALE denominator (5)")
_check("eog env+grade", _eog_env,   "all of EOG (2)")
_check("resources",     _resources, "the evolving modes (6)")
_check("react",         lambda: _harness_row(react_agent, model=LLM_MODEL, max_steps=1),
       "the ReAct harness -- your own loop (2) still works", deep=True)
_check("codex",         lambda: _harness_row(acp_codex_agent, max_episodes=1),
       "the Codex harness -- your own loop (2) still works", deep=True)
_check("ale",           _ale,       "all of ALE (5)", deep=True)

skip = [n for n, s in _rows if s == "SKIP"]
red = [n for n, s in _rows if s == "FAIL"]
ok = sum(1 for _, s in _rows if s == "OK")
line = f"\n{ok}/{len(_rows) - len(skip)} green"
line += f"   RED: {', '.join(red)}" if red else "   every feature on this deployment works"
print(line + (f"   (skipped: {', '.join(skip)})" if skip else ""))

## 1. What actually evolves

Before measuring the effect of harness growth, look at the growth itself. Stages are per domain
and they are **not** uniform: `calendar` has 3, `hr` has 5, `email` has 6. Each stage adds tasks
*and* enlarges the resource pool the agent is offered.

`resource_mode` controls which pool a task is provisioned with, and it is the knob the modes set
for you:

| `resource_mode` | the agent is offered | used by |
|---|---|---|
| `"oracle"` | only the capabilities this task actually needs | `task_specific` — the reference condition |
| `"accumulative"` | everything introduced up to this stage | `deployment_eval`, `self_evolving_adapt_eval` |
| `"none"` | no evolving resources at all | the floor |

`oracle ⊆ accumulative` always holds, and the gap between them is the distractor load — the
extra affordances the agent must search past. That gap is what grows.

In [ ]:
DOMAIN = "calendar"          # 3 stages: small enough to sweep, big enough to be a real matrix

versions = None
for ds in client.benchmarks()["datasets"]:
    if ds["dataset"] != "evovling_tools":
        continue
    for b in ds["benchmarks"]:
        if b["benchmark"] != "eog":
            continue
        for d in b["domains"]:
            if d["domain"] == DOMAIN:
                versions = sorted(d["versions"], key=lambda v: v["version"])

print(f"{DOMAIN}: {len(versions)} stages\n")
print(f"  {'stage':<7}{'train':>7}{'test':>7}{'oracle':>9}{'accumulative':>14}")
for v in versions:
    tid = client.task_ids("evovling_tools", "eog", v["version"], "test", DOMAIN)[0]
    counts = {m: client.resources("evovling_tools", "eog", v["version"], task_id=tid,
                                  split="test", domain=DOMAIN, mode=m)["count"]
              for m in ("oracle", "accumulative")}
    print(f"  v{v['version']:<6}{v['n_train']:>7}{v['n_test']:>7}"
          f"{counts['oracle']:>9}{counts['accumulative']:>14}")
print("\n^ the accumulative column is the harness growing; oracle is what the task truly needs.")

## 2. Size the sweep before you run it

Filling the matrix is not a flag you toggle idly. The last row alone is one pass over every
task; the full triangle re-evaluates every earlier cohort under every later harness, so it costs
roughly \(T\) times as much — and \(T\) differs per domain.

Work it out from the catalog first. `limit` is the release valve: it caps tasks per cohort, which
keeps the shape of the matrix while shrinking the bill, at the price of numbers that are no
longer comparable to anything.

In [ ]:
from simple_agentic_evals.benchmark import NON_PAPER_EOG_DOMAINS


def sweep_sizes(dataset="evovling_tools", benchmark="eog", split="test"):
    """Task-runs for the last row vs. the full triangle, per domain, from the catalog."""
    rows = []
    for ds in client.benchmarks()["datasets"]:
        if ds["dataset"] != dataset:
            continue
        for b in ds["benchmarks"]:
            if b["benchmark"] != benchmark:
                continue
            for d in b["domains"]:
                if d["domain"] in NON_PAPER_EOG_DOMAINS:   # not in any published table
                    continue
                n = [v[f"n_{split}"] for v in
                     sorted(d["versions"], key=lambda v: v["version"])]
                cum = tri = 0
                for x in n:                                 # sum of prefix sums = the triangle
                    cum += x
                    tri += cum
                rows.append((d["domain"], len(n), sum(n), tri))
    return rows


rows = sweep_sizes()
print(f"  {'domain':<24}{'stages':>7}{'last row':>10}{'matrix':>9}{'ratio':>7}")
for dom, T, last, tri in rows:
    print(f"  {str(dom):<24}{T:>7}{last:>10}{tri:>9}{tri / last:>7.1f}x")
tl, tt = sum(r[2] for r in rows), sum(r[3] for r in rows)
print(f"  {'TOTAL':<24}{'':>7}{tl:>10}{tt:>9}{tt / tl:>7.1f}x")
print(f"\n  matrix=True on the published scope is {tt:,} task-runs, not {tl:,}.")

## 3. Fill the matrix

One flag. `matrix=True` evaluates cohort \(\tau\) under harness \(t\) for every \(\tau \le t\),
instead of only the final pass.

The run below stays on one domain with `limit=2` so it finishes in minutes. That is enough to
produce a **real, correctly shaped matrix** — every cell is a genuine evaluation — while being far
too small for the cell values to mean anything. Treat the shape as the lesson and the numbers as
placeholders until you drop `limit`.

In [ ]:
def act(task, notes="", max_steps=6):
    """Level 1's MCP loop, plus room for whatever the policy has learned so far."""
    used = 0
    mcp = task.mcp_session(task.mcp_servers[0])
    try:
        tools = to_openai_tools(mcp.list_tools())
        system = task.system_prompt or ""
        if notes:
            system += "\n\nWhat you learned on earlier stages:\n" + notes
        msgs = [{"role": "system", "content": system},
                {"role": "user",   "content": task.user_prompt or ""}]
        for _ in range(max_steps):
            r = oai.chat.completions.create(model=LLM_MODEL, messages=msgs, tools=tools)
            used += r.usage.total_tokens
            m = r.choices[0].message
            msgs.append(m.model_dump(exclude_none=True))
            if not m.tool_calls:
                break
            for tc in m.tool_calls:
                out = mcp.call_tool(tc.function.name,
                                    json.loads(tc.function.arguments or "{}"))
                msgs.append({"role": "tool", "tool_call_id": tc.id,
                             "content": json.dumps(out)[:4000]})
    finally:
        mcp.close()
    return {"total_tokens": used}


def fixed_agent(task):
    """Deployment: the policy never changes, only the harness around it does."""
    return act(task)


deploy = run_benchmark(fixed_agent, "deployment_eval", client=client,
                       domain=DOMAIN, matrix=True, limit=2, progress=True)
print("\n" + str(deploy))

## 4. Read the matrix

`report.matrix` is a flat list of cells, each carrying `domain`, `stage` (the cohort \(\tau\)),
`at_stage` (the harness \(t\)) and the usual metrics. Pivoting it into the triangle is what makes
it legible.

One thing to know before you compare rows. `report.last_row` is **not** the bottom row of this
triangle. It is a separate, cheaper pass with `version="full"`, which pools every cohort at once
under the final harness — that is what ACC is computed from. The triangle's bottom row is the
same measurement taken cohort by cohort, which is what BWT needs. They answer the same question
at different resolutions, so expect them to agree closely rather than exactly.

In [ ]:
def print_matrix(report, domain=None, metric="accuracy"):
    """Pivot report.matrix into the paper's lower-triangular P[t][tau]."""
    cells = {(c.stage, c.at_stage): c for c in report.matrix
             if domain is None or c.domain == domain}
    if not cells:
        print("no matrix cells -- was the run made with matrix=True?")
        return
    stages = sorted({s for s, _ in cells} | {t for _, t in cells})
    print(f"  {'':<12}" + "".join(f"{'cohort v' + str(s):>12}" for s in stages))
    for t in stages:
        cs = "".join(f"{getattr(cells[(tau, t)], metric):>12.3f}" if (tau, t) in cells
                     else f"{'·':>12}" for tau in stages)
        print(f"  under H_v{t:<3}{cs}")
    print(f"\n  diagonal (adaptation): "
          + "  ".join(f"v{s}:{cells[(s, s)].accuracy:.3f}" for s in stages if (s, s) in cells))
    T = stages[-1]
    older = [s for s in stages[:-1] if (s, T) in cells and (s, s) in cells]
    if older:
        print("  retention (each earlier cohort, at introduction -> under the final harness):")
        for s in older:
            d = cells[(s, T)].accuracy - cells[(s, s)].accuracy
            print(f"     cohort v{s}   {cells[(s, s)].accuracy:.3f} -> "
                  f"{cells[(s, T)].accuracy:.3f}   ({d:+.3f})")


print_matrix(deploy, domain=DOMAIN)
print(f"\n  pooled last row (what ACC uses): {deploy.accuracy:.3f} over {deploy.n_tasks} tasks")

## 5. Methods that learn

`self_evolving_adapt_eval` adds one step. Before each stage is evaluated, the service hands your
method that stage's **training** split and lets it update whatever state it keeps:

```python
def adapt(stage, tasks):     # called once per stage, in order
    ...                      # update memory, rewrite a prompt, edit your own code
```

Pass it as `adapt=`, or just give your agent an `.adapt` method and `run_benchmark` finds it.
Omitting it is an error rather than a silent fallback — a self-evolving run without adaptation is
identical to a deployment run, and the comparison between them would be meaningless.

Two properties of `tasks` matter in practice. It is the **train** split, never the evaluation
tasks. And the prompts only exist once a task is provisioned, so reading them means entering the
task (`with t:`) — adaptation does real work and costs real time, which is why the published
rows report adaptation cost as part of their hours.

**Expect FWT near zero from the small run below.** With `limit=2` the same two tasks are
scored before and after each adaptation, and a shallow note rarely changes the outcome on two
tasks that were already failing the same way — so the before/after pairs often come out
identical. That is the sample size talking, not a broken measurement.

What your method does in there is the research question. The example below is deliberately
shallow — it accumulates task descriptions into a note the policy reads later. A memory method
would run the training tasks and distil what worked; a prompt optimizer would run a search like
GEPA's over them; a code-based method would rewrite its own harness.

In [ ]:
class EvolvingAgent:
    """A policy with state: `adapt` writes it, `__call__` reads it.

    run_benchmark picks up `.adapt` automatically, so this object is the whole
    integration -- no separate registration, no adapt= argument needed.
    """

    def __init__(self):
        self.notes: list[str] = []

    def adapt(self, stage, tasks):
        for t in tasks:
            with t:                       # prompts exist only once a task is provisioned
                self.notes.append(f"[v{stage}] {(t.user_prompt or '')[:140]}")

    def __call__(self, task):
        return act(task, notes="\n".join(self.notes[-8:]))


evolving = EvolvingAgent()
evolve = run_benchmark(evolving, "self_evolving_adapt_eval", client=client,
                       domain=DOMAIN, matrix=True, limit=2, progress=True)
print("\n" + str(evolve))
print(f"\nthe agent carried {len(evolving.notes)} notes out of adaptation")

## 6. BWT and FWT

Both are cohort-size-weighted means, exactly as the appendix defines them:

\[ \mathrm{BWT} = \frac{\sum_{i<T} n_i\,(P_{T,i} - P_{i,i})}{\sum_{i<T} n_i}
\qquad
\mathrm{FWT} = \frac{\sum_{i>1} n_i\,(P_{i,i} - P_{i,i}^{\text{before}})}{\sum_{i>1} n_i} \]

**BWT** compares each earlier cohort at the end against the same cohort at its introduction.
Negative means the growing harness cost you competence you already had — forgetting, whether
from distraction by a larger catalog or from stale accumulated state.

**FWT** compares a cohort under its own harness *after* that stage's adaptation against *before*
it. Negative means adaptation actively hurt the method's ability to use the capabilities that
stage introduced — over-specialization. This is why `before_adapt` cohorts are measured during
the sweep: once `adapt()` has run, the previous state is gone and the comparison is unavailable.
It is also why FWT is `None` in deployment mode — there is no adaptation to attribute it to.

Weighting matters. These are weighted by cohort size, so they do not equal the unweighted means
that `ContinualMetrics` reports in §7. A domain whose late stages are tiny will disagree
noticeably between the two conventions.

In [ ]:
print(f"  {'':<16}{'BWT':>9}{'FWT':>9}")
for name, r in (("deployment", deploy), ("self-evolving", evolve)):
    b = f"{r.bwt:+.3f}" if r.bwt is not None else "n/a"
    f = f"{r.fwt:+.3f}" if r.fwt is not None else "n/a"
    print(f"  {name:<16}{b:>9}{f:>9}")
print("  deployment has no FWT: with no adaptation there is nothing to attribute it to.")

print("\n  before/after adaptation on each new cohort (the FWT terms):")
before = {(c.domain, c.stage): c for c in evolve.before_adapt}
after = {(c.domain, c.stage): c for c in evolve.matrix if c.stage == c.at_stage}
for (dom, s), b in sorted(before.items(), key=lambda kv: kv[0][1]):
    a = after.get((dom, s))
    if a:
        print(f"     cohort v{s}   {b.accuracy:.3f} -> {a.accuracy:.3f}   "
              f"({a.accuracy - b.accuracy:+.3f})")

print("\n  for scale, published deployment rows on evovling_tools sit near BWT -0.05;")
print("  the paper's worst forgetting is -34.7% on ALE agents.")

## 7. Building the matrix yourself

`run_benchmark(matrix=True)` runs one trial per cell. If you need repeated trials per cell — the
published rows average 3 and report a spread — or you are scoring a system that does not fit the
`agent(task)` shape at all, build the matrix directly with `ContinualMetrics` and record cells as
you produce them.

It is the lower-level object: **0-based** stages, and `adapt_stage=-1` is the reserved key for a
cohort's before-adaptation baseline, which is what lets it compute FWT. It also reports
**forgetting** (best-minus-final per cohort), which the weighted path does not.

One caveat worth repeating: its BWT and FWT are *unweighted* means over cohorts, so they will not
match `report.bwt` / `report.fwt` unless every cohort happens to be the same size.

In [ ]:
from simple_agentic_evals import ContinualMetrics, StageResult

cm = ContinualMetrics(num_stages=3)

# Feed it whatever you measured -- here, the sweep from section 3, reindexed to 0-based.
for c in evolve.matrix:          # ONE run -- mixing runs would make FWT meaningless
    if c.domain != DOMAIN:
        continue
    cm.record(StageResult(eval_stage=c.stage - 1, adapt_stage=c.at_stage - 1,
                          num_tasks=c.n_tasks, success_rate=c.accuracy,
                          verifier_pass_rate=c.success_rate))

# The before-adaptation baselines FWT needs go in under adapt_stage=-1.
for c in evolve.before_adapt:
    if c.domain == DOMAIN:
        cm.record(StageResult(eval_stage=c.stage - 1, adapt_stage=-1,
                              num_tasks=c.n_tasks, success_rate=c.accuracy))

rep = cm.compute()
print(cm.print_report(rep))
print(f"\n  unweighted BWT {rep['BWT']:+.3f}  vs  weighted {evolve.bwt:+.3f}")
print("  they agree here only because `limit` made every cohort the same size;")
print("  drop it and the weighting starts to matter.")
print(f"  avg forgetting {rep['avg_forgetting']:+.3f}   max {rep['max_forgetting']:+.3f}")

## Cheat sheet

```python
report = run_benchmark(agent, "self_evolving_adapt_eval", client=client,
                       domain="calendar", matrix=True)
```

| argument | continual-learning meaning |
|---|---|
| `mode="deployment_eval"` | fixed method, growing harness. BWT only |
| `mode="self_evolving_adapt_eval"` | method adapts per stage. BWT **and** FWT; needs `adapt` |
| `mode="task_specific"` | oracle reference, single pass — `matrix=True` is an error here |
| `adapt=fn(stage, tasks)` | called once per stage with that stage's **train** split; or give the agent `.adapt` |
| `matrix=True` | fill the whole triangle, not just the last row. Costs ~\(T\)× |
| `limit=n` | keeps the matrix shape, shrinks the bill, forfeits comparability |

| on the report | |
|---|---|
| `.matrix` | every cell: `.domain`, `.stage` (cohort \(\tau\)), `.at_stage` (harness \(t\)) |
| `.before_adapt` | each cohort under its own harness *before* that stage adapted — the FWT baseline |
| `.last_row` | the pooled final pass ACC is computed from (not the triangle's bottom row) |
| `.bwt` `.fwt` | cohort-size-weighted, per the appendix. `.fwt` is `None` without adaptation |
| `.per_stage_acc` | the diagonal, by stage |

| reading the triangle | |
|---|---|
| diagonal \(P_{t,t}\) | adaptation — using capabilities as they arrive |
| below the diagonal | retention — old tasks under a bigger catalog |
| down a column | one cohort's story; the decline is forgetting |

**Next:** [Level 3 · Harnesses & modes](https://colab.research.google.com/drive/1mYQEDCVFStXMRWYI2hpEGBXwyRx1NSFj)
— the provided agents, every mode including standard non-evolving evaluation, and the full API
reference.